In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "main"
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 08 — Normalization

Stage 1.8. `build_numeric_scaler` builds `Log1p -> MinMax | ZScore` from the `scaling` block of
`config/config.yaml`. The scalers here are fitted on the full catalogue for the figures only;
the real fit happens in the 1.9 pipeline, on train data only.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recsys.io.artifact_store import ArtifactStore
from recsys.features.scaling import (
    WeightedCombinedDistance,
    build_numeric_scaler,
    compare_scalers,
)
from recsys.features.distance import JaccardDistance
from recsys.viz.plots import hist_box
from recsys.contracts import LOADED_RECIPES_SCHEMA, validate

store = ArtifactStore.from_config(CFG)
loaded = store.load("loaded")
recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)
SCALING_CFG = CFG.section("scaling")  # config/config.yaml `scaling` block

numeric_cols = [
    "minutes",
    "n_steps",
    "n_ingredients",
    "calories",
    "total_fat_pdv",
    "sugar_pdv",
    "sodium_pdv",
    "protein_pdv",
    "saturated_fat_pdv",
    "carbohydrates_pdv",
]
numeric_cols = [c for c in numeric_cols if c in recipes.columns]
print(f"Recipes: {recipes.shape} | Numeric cols: {len(numeric_cols)}")

In [ ]:
scaler_summary = compare_scalers(recipes, numeric_cols)
display(scaler_summary)

In [ ]:
# configured scaler (config `scaling` block) and the alternative, for the comparison only
mm_scaler = build_numeric_scaler({**SCALING_CFG, "method": "minmax"})
zs_scaler = build_numeric_scaler({**SCALING_CFG, "method": "zscore"})
configured_scaler = build_numeric_scaler(SCALING_CFG)
print("configured:", SCALING_CFG["method"], "| steps:", list(configured_scaler.named_steps))

raw_num = recipes[numeric_cols].apply(pd.to_numeric, errors="coerce").dropna()
mm_df = pd.DataFrame(mm_scaler.fit_transform(raw_num), columns=numeric_cols, index=raw_num.index)
zs_df = pd.DataFrame(zs_scaler.fit_transform(raw_num), columns=numeric_cols, index=raw_num.index)

figures = {
    "fig_08_01_calories_raw": hist_box(raw_num["calories"], title="Calories (Raw)"),
    "fig_08_02_calories_log1p_minmax": hist_box(mm_df["calories"], title="Calories (Log1p + MinMax)"),
    "fig_08_03_calories_log1p_zscore": hist_box(zs_df["calories"], title="Calories (Log1p + Z-Score)"),
    "fig_08_04_minutes_raw": hist_box(raw_num["minutes"], title="Minutes (Raw)"),
    "fig_08_05_minutes_log1p_minmax": hist_box(mm_df["minutes"], title="Minutes (Log1p + MinMax)"),
}
print(f"Generated {len(figures)} scaling figures.")

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

sample_recipes = recipes.loc[mm_df.index[:5]]
mlb = MultiLabelBinarizer(sparse_output=True)
X_ing_sample = mlb.fit_transform(sample_recipes["ingredients"])
X_num_sample = mm_df.loc[sample_recipes.index, ["minutes", "n_ingredients", "calories"]].to_numpy()

combined = WeightedCombinedDistance(
    ingredient_metric=JaccardDistance(), numeric_weight=SCALING_CFG["numeric_weight"]
)
D_sample = combined.pairwise((X_ing_sample, X_num_sample), (X_ing_sample, X_num_sample))
dist_df = pd.DataFrame(
    np.round(D_sample, 4),
    index=[f"recipe_{i}" for i in range(5)],
    columns=[f"recipe_{i}" for i in range(5)],
)
display(dist_df)

## Why the binary ingredient vector is not scaled

The ingredient vector is 0/1 (ingredient absent / present) and is compared with Jaccard or
cosine distance, which only look at shared vs. non-shared ingredients. Its values are already on
a fixed [0, 1] scale, so Min-Max would change nothing, and z-score would destroy sparsity
(every 0 becomes a non-zero negative number) and turn "absent" into a signal that depends on how
common an ingredient is. Scaling is therefore applied only to the numeric features (minutes,
n_steps, n_ingredients, nutrition), whose raw ranges differ by orders of magnitude; in
`WeightedCombinedDistance` the two parts are combined with an explicit weight
(`numeric_weight`) instead of letting one scale dominate.


In [ ]:
saved = store.save(
    stage="scaling",
    objects={
        "scaler_summary": scaler_summary,
        "combined_distance_sample": dist_df,
        **figures,
    },
    inputs=[loaded],
    params={
        "scaling_config": SCALING_CFG,
        "n_numeric_cols": len(numeric_cols),
    },
)

for fig in figures.values():
    plt.close(fig)

print("Saved scaling artifact:", saved.version)
print("\n=== SCALER SUMMARY (minutes & calories) ===")
print(
    scaler_summary[scaler_summary["feature"].isin(["minutes", "calories", "n_ingredients"])]
    .to_string(index=False)
)